In [1]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
import os
os.environ["OPENAI_API_KEY"] = "5a4a5c93bdc34d8caac98deac76ecc82."
os.environ["GAODE_API_KEY"] = ""

llm = ChatOpenAI(
    model = "glm-4.6",
    openai_api_base="https://open.bigmodel.cn/api/paas/v4/",
    extra_body={
        "thinking": {"type": "disabled"} 
    }
)

In [2]:
from pydantic import BaseModel

class TestUserIDRequest(BaseModel):
    user_id: str

from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 

@tool(args_schema=TestUserIDRequest)
def test_interrupt(user_id: str) -> str:
    """A tool to test agent interruption handling."""
    return f"User {user_id}, Tool executed successfully."

agent = create_agent(
    model=llm,
    tools=[test_interrupt],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "test_interrupt":True
        },
        description_prefix="正在测试代理中断处理"
    )],
    checkpointer=InMemorySaver()
)

In [7]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "some_id"}} 

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "我需要测试中断功能，请执行test_interrupt工具，用户ID 你随机生成一个。",
            }
        ]
    },
    config=config 
)

In [8]:
print(result.keys(), result['messages'][-1].content)

dict_keys(['messages', '__interrupt__']) 



In [ ]:
res = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "edit", "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)


中断功能测试已成功执行！

我使用随机生成的用户ID "user_test_78f4b2c9_2024" 调用了test_interrupt工具，工具执行成功。这表明系统中断处理功能正常工作。


In [24]:
print(''.join(message.content + "\n" for message in res['messages']))

我需要测试中断功能，请执行test_interrupt工具，用户ID 你随机生成一个。

我来帮您测试中断功能，我会为您生成一个随机的用户ID并执行测试。

User rejected the tool call for `test_interrupt` with id call_-8139803222562233967

我尝试执行了中断功能测试，使用了随机生成的用户ID "user_test_2024_001"，但是测试被用户拒绝了。

这可能表明中断处理功能正在正常工作 - 当用户拒绝工具调用时，系统能够正确响应并阻止执行。这正是中断功能的预期行为之一。

如果您希望重新测试中断功能，请告诉我，我可以使用另一个随机生成的用户ID再次尝试。
我需要测试中断功能，请执行test_interrupt工具，用户ID 你随机生成一个。


User user_test_78f4b2c9_2024, Tool executed successfully.

中断功能测试已成功执行！

我使用随机生成的用户ID "user_test_78f4b2c9_2024" 调用了test_interrupt工具，工具执行成功。这表明系统中断处理功能正常工作。

